# C6-pytorch — Practice p11 — Solution


**(a) One dense layer.**  The registered weight has shape
`(out, in)`: it has `out` rows, each containing `in` scalar weights,
for exactly $\text{out}\cdot\text{in}$ numbers.  The separately
registered bias has shape `(out,)`, so it contributes one scalar per
output unit, or $\text{out}$ more.  Adding the disjoint tensors gives
$\text{out}\cdot\text{in}+\text{out}
=\text{out}(\text{in}+1)$.

**(b) A chain.**  Layer $\ell$ maps $n_{\ell-1}$ inputs to $n_\ell$
outputs, so part (a) gives $n_\ell(n_{\ell-1}+1)$ registered scalars.
Each layer owns distinct parameter tensors, so the scalar counts add
across $\ell=1,\ldots,L$.  `ThresholdGate` registers neither a weight
nor a bias (indeed, no `nn.Parameter` at all), so every gate adds zero.
Therefore
$$P=\sum_{\ell=1}^{L}n_\ell(n_{\ell-1}+1).$$

**(c) Anchor.**  For $4\to7\to3$, the two contributions are
$7(4+1)=35$ and $3(7+1)=24$, hence $P=35+24=59$.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


first = DenseLayer(torch.zeros(7, 4), torch.zeros(7))
second = DenseLayer(torch.zeros(3, 7), torch.zeros(3))
formula_count = 7 * (4 + 1) + 3 * (7 + 1)
torch_count = sum(p.numel() for layer in (first, second) for p in layer.parameters())
anchor_gap = abs(formula_count - torch_count)

assert anchor_gap == 0
formula_count, torch_count, anchor_gap


### Answer check


In [ ]:
assert formula_count == 59
assert torch_count == 59
assert tuple(first.weight.shape) == (7, 4)
assert tuple(second.bias.shape) == (3,)
